# SpaceX Falcon 9 — Exploratory Data Analysis & Feature Engineering

## Project objective
Explore how launch sequence, payload mass, orbit, and launch site relate to Falcon 9 first-stage landing success. Then prepare model-ready features.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn.

**Target variable:** `Class` (1 = successful first-stage landing; 0 = unsuccessful first-stage landing).

> These charts describe associations in this dataset. They should not be interpreted as proof of causation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(DATA_URL)

required = {"FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Class", "Date"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Class"] = pd.to_numeric(df["Class"], errors="coerce")
df = df.dropna(subset=["Class", "Date"]).copy()
df["Class"] = df["Class"].astype(int)

print(f"Analysis rows: {len(df):,}")
print(f"Overall landing success rate: {df['Class'].mean():.1%}")
display(df.head())

## 1. Flight number and payload mass

Does landing success appear to change across launch sequence or payload mass? The hue distinguishes successful and unsuccessful landings.

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(data=df, x="FlightNumber", y="PayloadMass", hue="Class", alpha=0.8)
plt.title("Falcon 9 Landing Outcome by Flight Number and Payload Mass")
plt.xlabel("Flight number")
plt.ylabel("Payload mass (kg)")
plt.legend(title="Landing success", labels=["Unsuccessful (0)", "Successful (1)"])
plt.tight_layout()
plt.show()

## 2. Launch site and flight number

Compare the distribution of flight numbers across launch sites and landing outcomes.

In [ ]:
plt.figure(figsize=(11, 6))
sns.stripplot(data=df, x="FlightNumber", y="LaunchSite", hue="Class", dodge=True, alpha=0.75)
plt.title("Launch Site, Flight Number, and Landing Outcome")
plt.xlabel("Flight number")
plt.ylabel("Launch site")
plt.legend(title="Landing success", labels=["Unsuccessful (0)", "Successful (1)"])
plt.tight_layout()
plt.show()

## 3. Payload mass by launch site

Compare payload mass distributions at each launch site. This can help identify differences in the types of missions represented at each site.

In [ ]:
plt.figure(figsize=(11, 6))
sns.stripplot(data=df, x="PayloadMass", y="LaunchSite", hue="Class", dodge=True, alpha=0.7)
plt.title("Payload Mass by Launch Site and Landing Outcome")
plt.xlabel("Payload mass (kg)")
plt.ylabel("Launch site")
plt.legend(title="Landing success", labels=["Unsuccessful (0)", "Successful (1)"])
plt.tight_layout()
plt.show()

## 4. Landing success rate by orbit

A bar chart of the mean target value gives the observed landing success rate for each orbit. Include the number of launches so small groups are not overinterpreted.

In [ ]:
orbit_summary = (
    df.groupby("Orbit", dropna=False)
      .agg(launches=("Class", "size"), success_rate=("Class", "mean"))
      .sort_values("success_rate", ascending=False)
      .reset_index()
)
display(orbit_summary)

plt.figure(figsize=(12, 6))
sns.barplot(data=orbit_summary, x="Orbit", y="success_rate", color="steelblue")
plt.title("Observed First-Stage Landing Success Rate by Orbit")
plt.xlabel("Orbit")
plt.ylabel("Success rate")
plt.ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 5. Flight number and orbit

Inspect whether launch sequence and orbit type show visible patterns in landing outcomes.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df, x="FlightNumber", y="Orbit", hue="Class", alpha=0.8)
plt.title("Flight Number, Orbit, and Landing Outcome")
plt.xlabel("Flight number")
plt.ylabel("Orbit")
plt.legend(title="Landing success", labels=["Unsuccessful (0)", "Successful (1)"])
plt.tight_layout()
plt.show()

## 6. Payload mass and orbit

Compare payload mass across orbit categories, distinguishing successful and unsuccessful landings.

In [ ]:
plt.figure(figsize=(12, 7))
sns.stripplot(data=df, x="PayloadMass", y="Orbit", hue="Class", dodge=True, alpha=0.7)
plt.title("Payload Mass by Orbit and Landing Outcome")
plt.xlabel("Payload mass (kg)")
plt.ylabel("Orbit")
plt.legend(title="Landing success", labels=["Unsuccessful (0)", "Successful (1)"])
plt.tight_layout()
plt.show()

## 7. Yearly landing success trend

Aggregate by calendar year rather than plotting individual launch dates. Show launch counts alongside success rates to provide context.

In [ ]:
df["Year"] = df["Date"].dt.year
year_summary = (
    df.groupby("Year")
      .agg(launches=("Class", "size"), success_rate=("Class", "mean"))
      .reset_index()
)
display(year_summary)

fig, ax1 = plt.subplots(figsize=(11, 6))
sns.lineplot(data=year_summary, x="Year", y="success_rate", marker="o", ax=ax1)
ax1.set_title("Yearly Falcon 9 First-Stage Landing Success Rate")
ax1.set_xlabel("Year")
ax1.set_ylabel("Success rate")
ax1.set_ylim(0, 1)
ax1.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 8. Feature engineering

Select the features used in the next modeling stage. Keep the original DataFrame unchanged and create a separate feature table.

In [ ]:
feature_columns = [
    "FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Flights",
    "GridFins", "Reused", "Legs", "LandingPad", "Block",
    "ReusedCount", "Serial"
]
missing_features = set(feature_columns).difference(df.columns)
if missing_features:
    raise ValueError(f"Missing feature columns: {sorted(missing_features)}")

features = df[feature_columns].copy()
display(features.head())
print("Feature table shape:", features.shape)

## 9. Encode categorical features

One-hot encode categorical variables. Use `drop_first=False` so each category remains explicitly represented; downstream modeling can decide whether dimensionality reduction is needed.

In [ ]:
categorical_columns = ["Orbit", "LaunchSite", "LandingPad", "Serial"]
features_one_hot = pd.get_dummies(
    features,
    columns=categorical_columns,
    dtype=float
)

# Convert boolean columns to numeric values and ensure all features are numeric.
features_one_hot = features_one_hot.astype("float64")
display(features_one_hot.head())
print("Encoded feature table shape:", features_one_hot.shape)
print("All columns numeric:", all(pd.api.types.is_numeric_dtype(t) for t in features_one_hot.dtypes))

## 10. Export model features

Export the encoded features for the modeling notebook. The target variable is kept separate to avoid accidentally mixing labels into the feature matrix.

In [ ]:
target = df["Class"].astype("int64")
features_one_hot.to_csv("dataset_part_3.csv", index=False)
target.to_csv("dataset_target.csv", index=False, header=["Class"])

print("Saved dataset_part_3.csv and dataset_target.csv")
print(f"Feature rows: {len(features_one_hot):,} | Target rows: {len(target):,}")

## Findings and limitations

After running the notebook, summarize:
1. Which orbit categories have the highest observed success rates, and how many launches are in each group?
2. How does the success rate change over time?
3. Do patterns differ by launch site or payload mass?
4. Which features appear useful to investigate in a predictive model?

Do not infer causation from these plots. Small group sizes, changes in hardware over time, and the selected date range can affect observed rates.

## Reproducibility
- Source: IBM Skills Network SpaceX dataset (loaded in the first code cell).
- All charts and summary tables are generated from the loaded dataset.
- Run all cells from top to bottom to regenerate the feature exports.